In [58]:
from fastapi import FastAPI
from pydantic import BaseModel
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from dotenv import load_dotenv
import os

In [59]:
load_dotenv()

False

In [60]:
print("Loading Ollama Embeddings.It may take a few seconds or minutes ...")
# Initialize the embedding model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    dimensions=1024,
)

Loading Ollama Embeddings.It may take a few seconds or minutes ...


In [61]:
# We will store our FAISS database in this variable
vector_store = None
FAISS_INDEX_PATH = "faiss_index"

In [62]:
def process_and_embed(doc_id: int, provided_id: str, code: str, additional_info: str, text: str):
    # 1. Create Document (This creates 'doc' locally inside the function)
    doc = Document(
        page_content=text,
        metadata={
            "original_id": doc_id,
            "provided_id": provided_id,
            "raw_code": code,
            "additional_info": additional_info
        }
    )

    # 2. Split Document into chunks (Now it can read 'doc' because they are in the same scope!)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100
    )



    chunks = text_splitter.split_documents([doc])
    print(f"Split document {provided_id} into {len(chunks)} chunks.")


    # 3. Store chunks in FAISS
    if os.path.exists(FAISS_INDEX_PATH):
        vector_store = FAISS.load_local(
            FAISS_INDEX_PATH, embeddings,
            allow_dangerous_deserialization=True
        )
        vector_store.add_documents(chunks)
    else:
        vector_store = FAISS.from_documents(chunks, embeddings)

    vector_store.save_local(FAISS_INDEX_PATH)

    return len(chunks)

In [63]:
# 1. Run your modified function that returns the list of chunks
chunks_list = process_and_embed(
    doc_id=1,
    provided_id="TEST_001",
    code="def hello(): print('world')",
    additional_info="Test run data",
    text="This is the main processed text that will be chunked and embedded by Ollama."
)


Split document TEST_001 into 1 chunks.


In [64]:
# 2. Check how many total chunks were created
print(f"Total number of chunks: {chunks_list.__sizeof__()}")

# # 3. Loop through each chunk to see its text length and characters
# for i, chunk in enumerate(chunks_list):
#     print(f"\n--- Chunk {i + 1} ---")
#     print(f"Character Length: {len(chunk.page_content)}")
#     print(f"Content: {chunk.page_content}")
#     print(f"Metadata attached: {chunk.metadata}")

Total number of chunks: 28
